In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2

In [2]:
def get_poisson_error(r, conf_level=0.95):
    """
    Parameters
    ----------
    r : 1d numpy array filled with ints or floats
        rate (or count(s) ).
    conf_level : float, optional
        The decimal corresponding to the sigma interval.
        The default is 0.9 ( 90% conf. int. )

    Returns
    -------
    low_lim : float
        lower limit of the confidence interval.
    upp_lim : float
        upper limit of the confidence interval.
        
    Info
    -------
    If no bursts are detected, get single sided upper limit as rate
    new_conf_level = 1 - ( (1-conf_level)*2 ) # for single sided upper limit
    _, upp_lim = get_poisson_error(np.array([0]), conf_level=new_conf_level)

    """
    low_lim = 0.5 * chi2.ppf((1 - conf_level) / 2, 2 * r)
    low_lim[np.isnan(low_lim)] = 0
    upp_lim = 0.5 * chi2.ppf((1 + conf_level) / 2, 2 * (r+1))
    upp_lim[np.isnan(upp_lim)] = 0
    return low_lim, upp_lim

In [3]:
def get_rate_errors(bursts, conf_level=0.95):
    low_lim, upp_lim = get_poisson_error(bursts, conf_level=conf_level)
    # if no bursts are detected, get single sided upper limit as rate 
    lim = bursts == 0

    new_conf_level = 1 - ( (1-conf_level)*2 ) # for single sided upper limit 
    _, upp_lim[lim] = get_poisson_error(np.array([0]), conf_level=new_conf_level)
    
    dx_upp = upp_lim - bursts
    dx_low = bursts - low_lim

    return dx_low, dx_upp 

In [5]:
get_rate_errors(bursts=np.array([5]), conf_level=0.95)

(array([3.37651361]), array([6.66833208]))

In [6]:
def number_bursts(L_band=False, P_band=False):
    
    csv_r117 = "../dbs/FRB20220912A_table_paper.csv"
    
    db_r117 = pd.read_csv(csv_r117)
    db_r117["burst-n"] = db_r117['id'].str[:-3]
    
    #Drop the lowest fluence of a duplicate burst
    db_r117= db_r117.sort_values(by='fluence', ascending=False)
    db_r117= db_r117.drop_duplicates(subset=['burst-n'], keep="first")
    db_r117= db_r117.sort_values(by='toa', ascending=True)
    
    if L_band:
        db_r117 = db_r117[db_r117["central frequency"] > 400]
        db_r117_100 = db_r117[db_r117["fluence"] > 100]
        db_r117_500 = db_r117[db_r117["fluence"] > 500]
        db_r117_1000 = db_r117[db_r117["fluence"] > 1000]
        print("L-band")
        
    if P_band:
        db_r117 = db_r117[db_r117["central frequency"] < 400]
        db_r117_100 = db_r117[db_r117["fluence"] > 100]
        db_r117_500 = db_r117[db_r117["fluence"] > 500]
        db_r117_1000 = db_r117[db_r117["fluence"] > 1000]
        print("P-band")
        
    print(f"100:{len(db_r117_100)}, 500:{len(db_r117_500)}, 1000:{len(db_r117_1000)}")
    
    return

In [7]:
def number_burst_ata():
    
    csv_ata = "../dbs/allen-telescope/R117_allen_omar.csv"
    
    db_r117_ata = pd.read_csv(csv_ata)
    #display(db_r117_ata)
    
    db_r117_ata_100 = db_r117_ata[db_r117_ata["fluence"] > 100]
    db_r117_ata_500 = db_r117_ata[db_r117_ata["fluence"] > 500]
    db_r117_ata_1000 = db_r117_ata[db_r117_ata["fluence"] > 1000]
    
    print("ATA")
    print(f"100:{len(db_r117_ata_100)}, 500:{len(db_r117_ata_500)}, 1000:{len(db_r117_ata_1000)}")
    
    return
number_burst_ata()

ATA
100:18, 500:5, 1000:2


In [8]:
number_bursts(L_band=True)

L-band
100:48, 500:13, 1000:4


In [9]:
number_bursts(P_band=True)

P-band
100:16, 500:7, 1000:3


In [10]:
def det_rate_err(N, time, fluence_limit, band):

    time_d = time / 24
    number_bursts = N
    
    number_bursts_err = np.sqrt(N)
    
    det_rate = N / time_d
    
    det_rate_err_lw = get_rate_errors(bursts=np.array([number_bursts]), conf_level=0.95)[0][0]/time_d
    det_rate_err_up = get_rate_errors(bursts=np.array([number_bursts]), conf_level=0.95)[1][0]/time_d

    print(f"{fluence_limit} Jy ms - {band}: {det_rate:.2f} +{det_rate_err_up:.2f} -{det_rate_err_lw:.2f} burst/day")
    
    return 

In [11]:
def det_rate_func():
    "The detection rate for L- and P-band during our campaign"
    det_rate_err(N=48, time=1158.57, fluence_limit=100, band="L")
    det_rate_err(N=16, time=607.34, fluence_limit=100, band="P")
    print("------")

    det_rate_err(N=13, time=1158.57, fluence_limit=500, band="L")
    det_rate_err(N=7, time=607.34, fluence_limit=500, band="P")
    print("------")
    
    det_rate_err(N=4, time=1158.57, fluence_limit=1000, band="L")
    det_rate_err(N=3, time=607.34, fluence_limit=1000, band="P")
    print("------")
    
det_rate_func()

100 Jy ms - L: 0.99 +0.32 -0.26 burst/day
100 Jy ms - P: 0.63 +0.39 -0.27 burst/day
------
500 Jy ms - L: 0.27 +0.19 -0.13 burst/day
500 Jy ms - P: 0.28 +0.29 -0.17 burst/day
------
1000 Jy ms - L: 0.08 +0.13 -0.06 burst/day
1000 Jy ms - P: 0.12 +0.23 -0.09 burst/day
------


In [12]:
def skyrate_cald(N, hrs, fluence_threshold):
    
    fractional_year = hrs / 8760.
    total_exposure = 5.1e5  # deg^2*hr; ASKAP survey, Shannon et al. 2018, Nature
    
    #Scaling the sky rate to the fluence threshold
    sky_rate_fluence_threshold = 5e3 / (100/fluence_threshold)**(-1.5) # /sky/yr above Fluence threshold
#     skyrate_100 = 5e3  
#     skyrate_500 = skyrate_100 / (1./5.)**(-1.5)
#     skyrate_1000 = skyrate_100 / (1./10.)**(-1.5)
    
    fraction_nbursts_yr = N / fractional_year
    frac_nbursts_yr_err_lw = get_rate_errors(bursts=np.array([N]), conf_level=0.95)[0][0]/fractional_year
    frac_nbursts_yr_err_up = get_rate_errors(bursts=np.array([N]), conf_level=0.95)[1][0]/fractional_year
        
    fraction_nbursts_yr_rate_pc = round(fraction_nbursts_yr / sky_rate_fluence_threshold * 100,1)
    
    frac_nbursts_yr_rate_lw = round(frac_nbursts_yr_err_lw / sky_rate_fluence_threshold * 100,1)
    frac_nbursts_yr_rate_up = round(frac_nbursts_yr_err_up / sky_rate_fluence_threshold * 100,1)
        
    print(f'We detected {round(fraction_nbursts_yr,1)} bursts/yr above {fluence_threshold} Jy ms')
    print(f'That corresponds to {fraction_nbursts_yr_rate_pc}% +{frac_nbursts_yr_rate_up}% -{frac_nbursts_yr_rate_lw}% of bursts')
    print("---------")
    return

In [13]:
def skyrate(L_band=False,r67=False,ATA=False):
    """
    Following a similiar method employed by Kirsten et al (2024)
    github link: https://github.com/astroflash-frb/frb20201124A-kirsten-2023/blob/main/python/skyrates.py 
    
    The hrs_observed for L-band are the total unique observing hours we have, notice the unique. 
    Number of bursts are from the previous function, taken duplicates into account
    
    In order to check the methods -- the boolean r67 reproduces the numbers used in kirsten et al 2024.
    Those are the same numbers, so the method checks out.
    """
    
    if L_band == True:
        hrs_observed = 1158.57
        nbursts_100 = 48  # bursts with fluence >= 100 Jyms 
        nbursts_500 = 13   # burst with fluence >= 500 Jyms 
        nbursts_1000 = 4 # burst with fluence >= 1000 Jyms 
        print("-------L-Band - R117 --------")
        
    
    #From Kirsten et al. 2024 -- To be able to reproduce the same results
    if r67 == True:
        hrs_observed = 2281
        nbursts_100 = 17  # bursts with fluence >= 100 Jyms 
        nbursts_500 = 3   # burst with fluence >= 500 Jyms 
        nbursts_1000 = 1   # burst with fluence >= 1000 Jyms 
        print("-------R67-results--------")

    #From Sheikh et al. 2024 -- They quote above 100 Jy ms, a few perecent, to calculate it myself
    if ATA == True:
        hrs_observed = 541
        nbursts_100 = 18  # bursts with fluence >= 100 Jyms 
        nbursts_500 = 5   # burst with fluence >= 500 Jyms 
        nbursts_1000 = 2   # burst with fluence >= 1000 Jyms 
        print("-------ATA-Sheikh (2024)--------")
    
    print(f"based on {hrs_observed} observing hours")
    print(f"100 Jy ms: {nbursts_100}, 500 Jy ms: {nbursts_500}, 1000 Jy ms: {nbursts_1000}")
    print("---------")
    
    skyrate_cald(N=nbursts_100, hrs=hrs_observed, fluence_threshold=100)
    skyrate_cald(N=nbursts_500, hrs=hrs_observed, fluence_threshold=500)
    skyrate_cald(N=nbursts_1000, hrs=hrs_observed, fluence_threshold=1000)

    return

In [14]:
skyrate(r67=True)

-------R67-results--------
based on 2281 observing hours
100 Jy ms: 17, 500 Jy ms: 3, 1000 Jy ms: 1
---------
We detected 65.3 bursts/yr above 100 Jy ms
That corresponds to 1.3% +0.8% -0.5% of bursts
---------
We detected 11.5 bursts/yr above 500 Jy ms
That corresponds to 2.6% +5.0% -2.0% of bursts
---------
We detected 3.8 bursts/yr above 1000 Jy ms
That corresponds to 2.4% +11.1% -2.4% of bursts
---------


In [15]:
skyrate(L_band=True)

-------L-Band - R117 --------
based on 1158.57 observing hours
100 Jy ms: 48, 500 Jy ms: 13, 1000 Jy ms: 4
---------
We detected 362.9 bursts/yr above 100 Jy ms
That corresponds to 7.3% +2.4% -1.9% of bursts
---------
We detected 98.3 bursts/yr above 500 Jy ms
That corresponds to 22.0% +15.6% -10.3% of bursts
---------
We detected 30.2 bursts/yr above 1000 Jy ms
That corresponds to 19.1% +29.8% -13.9% of bursts
---------


In [16]:
skyrate(ATA=True)

-------ATA-Sheikh (2024)--------
based on 541 observing hours
100 Jy ms: 18, 500 Jy ms: 5, 1000 Jy ms: 2
---------
We detected 291.5 bursts/yr above 100 Jy ms
That corresponds to 5.8% +3.4% -2.4% of bursts
---------
We detected 81.0 bursts/yr above 500 Jy ms
That corresponds to 18.1% +24.1% -12.2% of bursts
---------
We detected 32.4 bursts/yr above 1000 Jy ms
That corresponds to 20.5% +53.5% -18.0% of bursts
---------
